# Assignment 4 — Árboles de Decisión: Clasificación de Vinos**Unidad 2 | Assignment 4**> versión: 2025-1  |  modificado: 2026-04-19> 📝 **Modalidad:** Trabajo autónomo — practica a tu ritmo.

<div style="background-color:#F8F9FA; border:2px solid #AEB6BF; padding:12px 18px; border-radius:8px; margin:10px 0;"><strong>🎓 Modo de uso:</strong> Este notebook es compartido por dos cursos.<br><br><span style="color:#2E86C1; font-weight:bold;">🔵 Pregrado</span> — Trabaja el contenido general y los bloques azules. Los bloques amarillos son opcionales y te darán contexto adicional.<br><br><span style="color:#B7950B; font-weight:bold;">🟡 Doctorado</span> — Trabaja el contenido general y <em>ambos</em> bloques. Los bloques azules te ofrecen la intuición; los amarillos, la formalización.</div>

## Rúbrica orientativa<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1; padding:14px 18px; border-radius:6px; margin:12px 0;"><span style="color:#2E86C1; font-weight:bold; font-size:1em;">🔵 Pregrado — Competencias esperadas</span><br><br><ul style="color:#2E86C1;"><li><strong>Exploración:</strong> Cargar dataset, describir features, visualizar distribución de clases.</li><li><strong>Entrenamiento:</strong> Entrenar un árbol de decisión con parámetros por defecto y con búsqueda de hiperparámetros (max_depth).</li><li><strong>Evaluación:</strong> Calcular accuracy, F1-macro, matriz de confusión. Interpretar resultados en contexto.</li><li><strong>Visualización:</strong> Graficar el árbol y sus feature importances.</li><li><strong>Comparación:</strong> Comparar árbol con Gaussian NB y LDA usando validación cruzada.</li><li><strong>Reflexión:</strong> Responder preguntas sobre interpretabilidad, overfitting y aplicabilidad.</li></ul><br><span style="font-size:0.85em; color:#5D6D7E;">→ ¿Quieres ver la formalización matemática y análisis avanzado? Continúa en el bloque 🟡 Doctorado.</span></div>

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D; padding:14px 18px; border-radius:6px; margin:12px 0;"><span style="color:#B7950B; font-weight:bold; font-size:1em;">🟡 Doctorado — Competencias esperadas (ADEMÁS de pregrado)</span><br><br><ul style="color:#B7950B;"><li><strong>Análisis de hiperparámetros:</strong> Estudiar sensibilidad a max_depth, min_samples_split, min_samples_leaf con heatmaps.</li><li><strong>Importancia de features:</strong> Calcular permutation importance vs MDI importance. Análisis crítico de sesgos.</li><li><strong>Fundamentos:</strong> Derivar analíticamente impureza Gini del nodo raíz. Verificar con métricas del modelo.</li><li><strong>Optimalidad:</strong> Diseñar experimento que demuestre limitaciones del algoritmo greedy de CART.</li><li><strong>Teoría:</strong> Análisis del límite de Bayes: ¿cuándo árboles profundos/superficiales son apropiados?</li></ul><br><span style="font-size:0.85em; color:#7D6608;">→ La intuición detrás de estas derivaciones está en el bloque 🔵 Pregrado.</span></div>---**Nota:** Esta rúbrica es orientativa. La nota final se asigna en la prueba escrita, no en este notebook.

## Contexto del problema### Dataset: Wine (UCI Machine Learning Repository)**Descripción:**- **Muestras:** 178 vinos cultivados en Italia- **Features:** 13 mediciones químicas (alcohol, acidez, fenoles, etc.)- **Clases:** 3 tipos de vino (cultivares distintos)- **Tarea:** Clasificación multiclase — predecir el tipo de vino a partir de sus características químicas### Objetivo de este assignmentEntrenarás un **árbol de decisión** para clasificar el tipo de vino. Explorarás cómo el árbol toma decisiones (¿qué feature usa para dividir primero?), cómo evitar overfitting mediante la optimización de hiperparámetros, y compararás el árbol con modelos generativos como Gaussian NB y LDA.### Restricciones⚠️ **No usar ensemble methods** (Random Forests, Boosting, etc.). Aún no llegamos a esa unidad. Nos enfocamos en árboles simples.

In [ ]:
# ============================================# Setup (NO MODIFICAR)# ============================================# dataset: wine (sklearn built-in)  |  generado sintéticamente: noimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.datasets import load_wine, load_breast_cancerfrom sklearn.tree import DecisionTreeClassifier, plot_treefrom sklearn.model_selection import train_test_split, GridSearchCV, cross_val_scorefrom sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrixfrom sklearn.naive_bayes import GaussianNBfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysisfrom sklearn.inspection import permutation_importanceimport warningswarnings.filterwarnings('ignore')# Fijar seeds para reproducibilidadRANDOM_STATE = 42np.random.seed(RANDOM_STATE)# Estilossns.set_style("whitegrid")plt.rcParams['figure.figsize'] = (12, 6)print("✅ Setup completo")

---## Sección A: Exploración y Preprocesamiento### 1. Cargar y explorar el dataset

In [ ]:
# Cargar dataset Winewine = load_wine()X = pd.DataFrame(wine.data, columns=wine.feature_names)y = pd.Series(wine.target, name='target')print("Dataset Wine:")print(f"  Muestras: {X.shape[0]}")print(f"  Features: {X.shape[1]}")print(f"  Clases: {len(np.unique(y))}")print(f"\nDistribución de clases:")print(y.value_counts().sort_index())print(f"\nPrimeras 5 muestras:")print(X.head())

In [ ]:
# 🔍 Tests de sanidad — verifica tu trabajotry:    assert X.shape[0] == 178, "El dataset debe tener 178 muestras"    assert X.shape[1] == 13, "El dataset debe tener 13 features"    assert y.nunique() == 3, "Debe haber 3 clases"    assert y.isnull().sum() == 0, "No debe haber valores faltantes en y"    assert X.isnull().sum().sum() == 0, "No debe haber valores faltantes en X"    print("✅ Tests de sanidad pasados: EDA correcta")except AssertionError as e:    print(f"❌ Error: {e}")

In [ ]:
# Train/Test split estratificado (75/25)X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)print(f"Conjunto de entrenamiento: {X_train.shape[0]} muestras")print(f"Conjunto de prueba: {X_test.shape[0]} muestras")print(f"\nDistribución en train:")print(y_train.value_counts().sort_index())print(f"\nDistribución en test:")print(y_test.value_counts().sort_index())

In [ ]:
# 🔍 Tests de sanidad — verifica tu trabajotry:    assert X_train.shape[0] + X_test.shape[0] == 178, "Split debe preservar total de muestras"    assert abs(X_train.shape[0] / 178 - 0.75) < 0.01, "Split no es aproximadamente 75/25"    assert y_train.nunique() == 3 and y_test.nunique() == 3, "Ambos conjuntos deben tener las 3 clases"    print("✅ Tests de sanidad pasados: split correcto")except AssertionError as e:    print(f"❌ Error: {e}")

---## Sección B: Modelo Base y Evaluación### 1. Entrenar árbol de decisión con parámetros por defecto

In [ ]:
# Entrenar árbol con parámetros por defectotree_default = DecisionTreeClassifier(random_state=RANDOM_STATE)tree_default.fit(X_train, y_train)# Prediccionesy_pred_train = tree_default.predict(X_train)y_pred_test = tree_default.predict(X_test)# Evaluartrain_acc = accuracy_score(y_train, y_pred_train)test_acc = accuracy_score(y_test, y_pred_test)train_f1 = f1_score(y_train, y_pred_train, average='macro')test_f1 = f1_score(y_test, y_pred_test, average='macro')print("=== Árbol de Decisión (parámetros por defecto) ===")print(f"\nAccuracy:")print(f"  Train: {train_acc:.4f}")print(f"  Test:  {test_acc:.4f}")print(f"\nF1-macro:")print(f"  Train: {train_f1:.4f}")print(f"  Test:  {test_f1:.4f}")print(f"\nProfundidad del árbol: {tree_default.get_depth()}")print(f"Número de hojas: {tree_default.get_n_leaves()}")print(f"\n=== Classification Report (Test Set) ===")print(classification_report(y_test, y_pred_test, target_names=wine.target_names))

In [ ]:
# Visualizar matriz de confusióncm = confusion_matrix(y_test, y_pred_test)plt.figure(figsize=(8, 6))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,            xticklabels=wine.target_names, yticklabels=wine.target_names)plt.title('Matriz de Confusión — Árbol de Decisión (Test Set)')plt.ylabel('Verdadero')plt.xlabel('Predicho')plt.tight_layout()plt.show()# Interpretaciónprint("\n📊 Interpretación de la matriz:")print(f"  ✓ Aciertos en diagonal (predic correctas)")print(f"  ✗ Errores fuera de diagonal")

In [ ]:
# 🔍 Tests de sanidad — verifica tu trabajotry:    assert tree_default.get_depth() > 0, "El árbol debe tener profundidad > 0"    assert 0 <= train_acc <= 1 and 0 <= test_acc <= 1, "Accuracies deben estar en [0,1]"    assert 0 <= train_f1 <= 1 and 0 <= test_f1 <= 1, "F1 scores deben estar en [0,1]"    assert cm.shape == (3, 3), "Matriz de confusión debe ser 3x3"    print("✅ Tests de sanidad pasados: modelo base correcto")except AssertionError as e:    print(f"❌ Error: {e}")

---## Sección C: Visualización del Árbol y Feature Importance### 1. Visualizar el árbol (versión simplificada)

In [ ]:
# Visualizar árbol con max_depth=3 para claridadfig, ax = plt.subplots(figsize=(20, 10))plot_tree(tree_default, feature_names=wine.feature_names, class_names=wine.target_names,          ax=ax, filled=True, fontsize=10, max_depth=3)plt.title('Árbol de Decisión (primeros 3 niveles) — Wine Dataset', fontsize=14)plt.tight_layout()plt.show()print("Nota: El árbol completo tiene más niveles. Se visualizan aquí los primeros 3 para claridad.")

In [ ]:
# Feature importances (Mean Decrease in Impurity)feature_importance = pd.DataFrame({    'feature': wine.feature_names,    'importance': tree_default.feature_importances_}).sort_values('importance', ascending=False)# Top 10top_10 = feature_importance.head(10)plt.figure(figsize=(10, 6))sns.barplot(data=top_10, x='importance', y='feature', palette='viridis')plt.title('Top 10 Features — Mean Decrease in Impurity (MDI)', fontsize=12)plt.xlabel('Importancia')plt.ylabel('Feature')plt.tight_layout()plt.show()print("Top 10 features:")print(top_10.to_string(index=False))

In [ ]:
# 🔍 Tests de sanidad — verifica tu trabajotry:    assert len(tree_default.feature_importances_) == 13, "Debe haber importancia para 13 features"    assert np.isclose(tree_default.feature_importances_.sum(), 1.0, atol=1e-6), "Importancias deben sumar a 1"    assert np.all(tree_default.feature_importances_ >= 0), "Importancias no pueden ser negativas"    print("✅ Tests de sanidad pasados: feature importances correcto")except AssertionError as e:    print(f"❌ Error: {e}")

---## Sección D: Optimización y Comparación con otros modelos<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1; padding:14px 18px; border-radius:6px; margin:12px 0;"><span style="color:#2E86C1; font-weight:bold; font-size:1em;">🔵 Pregrado</span><br><br>### D.1. Búsqueda de max_depth óptimoEl parámetro **max_depth** controla la profundidad máxima del árbol:- **max_depth bajo** → árbol más simple, menos overfitting, posible underfitting- **max_depth alto** → árbol complejo, más overfitting, mejor ajuste a train**Tarea:** Usa `GridSearchCV` para buscar el max_depth óptimo en el rango [2, 3, 4, 5, 6, 7, 8]. Metrica de evaluación: F1-macro con 5-fold cross-validation.**TODO 1:** Completa el código para entrenar el árbol optimizado.</div>

In [ ]:
# TODO 1: Buscar max_depth óptimo con GridSearchCV# Completar el código siguienteparam_grid = {'max_depth': [2, 3, 4, 5, 6, 7, 8]}tree_gs = GridSearchCV(    DecisionTreeClassifier(random_state=RANDOM_STATE),    param_grid=param_grid,    cv=5,    scoring='f1_macro',    n_jobs=-1)# Entrenartree_gs.fit(X_train, y_train)# Mejor parámetrobest_depth = tree_gs.best_params_['max_depth']best_cv_score = tree_gs.best_score_print(f"Best max_depth: {best_depth}")print(f"Best CV F1-macro: {best_cv_score:.4f}")# Entrenar modelo optimizadotree_optimized = tree_gs.best_estimator_y_pred_opt_test = tree_optimized.predict(X_test)acc_opt_test = accuracy_score(y_test, y_pred_opt_test)f1_opt_test = f1_score(y_test, y_pred_opt_test, average='macro')print(f"\nÁrbol optimizado:")print(f"  Accuracy test: {acc_opt_test:.4f}")print(f"  F1-macro test: {f1_opt_test:.4f}")print(f"  Profundidad: {tree_optimized.get_depth()}")

### D.2. Comparación: Árbol vs Gaussian NB vs LDAComparemos el árbol con dos modelos que estudiaste en la unidad anterior:- **Gaussian Naive Bayes (NB):** modelo generativo simple- **Linear Discriminant Analysis (LDA):** modelo discriminativo lineal**TODO 2:** Entrenar ambos modelos y comparar con el árbol usando validación cruzada.

In [ ]:
# TODO 2: Entrenar Gaussian NB y LDA, comparar con árbol# Entrenar modelosnb_model = GaussianNB()lda_model = LinearDiscriminantAnalysis()# Validación cruzada (5-fold) — F1-macrocv_folds = 5cv_tree = cross_val_score(    DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE),    X_train, y_train, cv=cv_folds, scoring='f1_macro')cv_nb = cross_val_score(nb_model, X_train, y_train, cv=cv_folds, scoring='f1_macro')cv_lda = cross_val_score(lda_model, X_train, y_train, cv=cv_folds, scoring='f1_macro')# Resultadoscomparison = pd.DataFrame({    'Model': ['Decision Tree', 'Gaussian NB', 'LDA'],    'CV Mean F1': [cv_tree.mean(), cv_nb.mean(), cv_lda.mean()],    'CV Std F1': [cv_tree.std(), cv_nb.std(), cv_lda.std()]})print("=== Comparación con validación cruzada (F1-macro) ===")print(comparison.to_string(index=False))# Visualizarfig, ax = plt.subplots(figsize=(10, 5))models = comparison['Model']means = comparison['CV Mean F1']stds = comparison['CV Std F1']ax.bar(models, means, yerr=stds, capsize=5, alpha=0.7, color=['#3498db', '#e74c3c', '#2ecc71'])ax.set_ylabel('F1-macro')ax.set_title('Comparación de Modelos (5-fold CV)', fontsize=12)ax.set_ylim([0.8, 1.0])plt.tight_layout()plt.show()

### D.3. Preguntas de reflexión aplicada**Pregunta 1:** Si un sommelier te pregunta *"¿por qué clasificaste este vino como tipo 2?"*, ¿cómo usarías el árbol para responderle de forma interpretable?<\!-- RESPUESTA MODELO:El árbol de decisión es completamente interpretable. Puedes seguir el camino desde la raíz hasta la hoja que predicó tipo 2, y explicar cada split: "primero verificamos si [feature A] < umbral1, luego si [feature B] >= umbral2, etc., y por eso predecimos tipo 2". Esto es imposible en una red neuronal (caja negra) o en Naive Bayes (muchas probabilidades simultáneas).-->**Pregunta 2:** ¿Por qué el árbol con parámetros por defecto (sin limitar max_depth) podría estar haciendo overfitting?<\!-- RESPUESTA MODELO:Sin restricción de profundidad, el árbol crece hasta los límites: puede tener hojas muy profundas con muy pocas muestras (incluso 1 muestra por hoja), memorizando el conjunto de entrenamiento. Esto explica por qué el accuracy en train es frecuentemente 1.0 pero el test es más bajo. La diferencia train-test es el gap de overfitting.-->**Pregunta 3:** ¿Qué feature química es más discriminativa (aparece en splits superiores del árbol)? ¿Tiene sentido enológico?<\!-- RESPUESTA MODELO:Mira el feature_importance y el árbol visualizado. Es común que "alcohol" o "phenols" sean muy discriminativos, ya que son características centrales de diferenciación entre cultivares. Esto tiene sentido: diferentes cultivos tienen diferentes perfiles químicos.-->

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D; padding:14px 18px; border-radius:6px; margin:12px 0;"><span style="color:#B7950B; font-weight:bold; font-size:1em;">🟡 Doctorado</span><br><br>## Sección E: Análisis Avanzado — Hiperparámetros y Fundamentos Teóricos### E.1. Sensibilidad a hiperparámetros: Heatmap de GridSearchCVAdemás del max_depth, CART tiene otros hiperparámetros críticos:- **min_samples_split:** mínimo número de muestras para dividir un nodo- **min_samples_leaf:** mínimo número de muestras en una hoja**TODO 3:** Construir una grilla 2D (min_samples_split vs min_samples_leaf) y visualizar con heatmap el score F1-macro.</div>

In [ ]:
# [PhD]# TODO 3: Heatmap de sensibilidad a hiperparámetrosmin_samples_split_range = [2, 5, 10, 15, 20]min_samples_leaf_range = [1, 2, 4, 6, 8]results = np.zeros((len(min_samples_split_range), len(min_samples_leaf_range)))for i, mss in enumerate(min_samples_split_range):    for j, msl in enumerate(min_samples_leaf_range):        tree = DecisionTreeClassifier(            max_depth=best_depth,            min_samples_split=mss,            min_samples_leaf=msl,            random_state=RANDOM_STATE        )        scores = cross_val_score(tree, X_train, y_train, cv=5, scoring='f1_macro')        results[i, j] = scores.mean()# Visualizarfig, ax = plt.subplots(figsize=(10, 6))sns.heatmap(results, annot=True, fmt='.3f', cmap='YlGnBu', cbar_kws={'label': 'F1-macro'},            xticklabels=min_samples_leaf_range, yticklabels=min_samples_split_range, ax=ax)ax.set_xlabel('min_samples_leaf')ax.set_ylabel('min_samples_split')ax.set_title('Sensibilidad a hiperparámetros (5-fold CV)', fontsize=12)plt.tight_layout()plt.show()print("Heatmap de CV scores completado.")

### E.2. Permutation Importance vs Mean Decrease in Impurity**Problema:** Mean Decrease in Impurity (MDI) tiene un sesgo hacia features de alta cardinalidad continua.**Solución:** Permutation Importance mide la caída en métrica cuando permutamos una feature.**TODO 4:** Calcular permutation_importance y comparar con MDI.

In [ ]:
# [PhD]# TODO 4: Comparar MDI vs Permutation Importance# Permutation importanceperm_importance = permutation_importance(    tree_optimized, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)perm_df = pd.DataFrame({    'feature': wine.feature_names,    'permutation_importance': perm_importance.importances_mean,    'mdi_importance': tree_optimized.feature_importances_}).sort_values('permutation_importance', ascending=False)# Top 10top_10_perm = perm_df.head(10)# Visualizar ambasfig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].barh(range(10), top_10_perm['mdi_importance'], color='#3498db')axes[0].set_yticks(range(10))axes[0].set_yticklabels(top_10_perm['feature'])axes[0].set_xlabel('Importancia')axes[0].set_title('MDI Importance', fontsize=12)axes[1].barh(range(10), top_10_perm['permutation_importance'], color='#e74c3c')axes[1].set_yticks(range(10))axes[1].set_yticklabels(top_10_perm['feature'])axes[1].set_xlabel('Importancia')axes[1].set_title('Permutation Importance', fontsize=12)plt.tight_layout()plt.show()print("\nTop 10 features — Comparación MDI vs Permutation:")print(top_10_perm.to_string(index=False))

### E.3. Derivación analítica: Impureza Gini del nodo raízCART usa el índice de Gini como criterio de split:$$\text{Gini}(D) = 1 - \sum_{k=1}^{K} \left( \frac{n_k}{n} \right)^2$$donde $n_k$ es el número de muestras de clase $k$ y $n$ es el total.**TODO 5:** Calcular la impureza Gini del nodo raíz analíticamente y verificar que coincida con los valores del árbol.<\!-- RESPUESTA MODELO:La impureza Gini se calcula en el split raíz. Si tenemos 59, 71 y 48 muestras en las 3 clases:Gini = 1 - (59/178)^2 - (71/178)^2 - (48/178)^2 = 1 - 0.1098 - 0.1594 - 0.0728 ≈ 0.658Este valor está en el árbol visualizado. El algoritmo greedy busca el split que maximiza la ganancia de información.-->

In [ ]:
# [PhD]# TODO 5: Derivar analíticamente la impureza Gini del nodo raíz# Contar muestras por clase en X_trainclass_counts = np.bincount(y_train)n_total = len(y_train)print(f"Distribución en training set:")for c, count in enumerate(class_counts):    print(f"  Clase {c}: {count} muestras")# Calcular Ginigini_root = 1 - np.sum((class_counts / n_total) ** 2)print(f"\nImpureza Gini (nodo raíz) — cálculo analítico:")print(f"  Gini = 1 - (59/178)^2 - (71/178)^2 - (48/178)^2")print(f"  Gini = {gini_root:.4f}")# Verificar con el árbolprint(f"\nVerificación: Gini en el árbol (tree_optimized):")print(f"  (Está embebido en el árbol visualizado)")# Calcular ganancia en el mejor split (raíz)# El árbol usa este Gini para seleccionar el mejor feature y thresholdprint(f"\n✓ La impureza Gini del nodo raíz se usa para calcular la ganancia de información")print(f"  en cada split. Un algoritmo greedy elige el split que maximiza IG = Gini_parent - weighted(Gini_children)")

### E.4. Optimalidad de CART: Limitaciones del algoritmo greedyEl algoritmo CART es **greedy**: en cada nodo, elige el split óptimo localmente, sin garantizar optimalidad global.**TODO 6:** Diseña un experimento simple que demuestre que el árbol entrenado NO es globalmente óptimo.<\!-- RESPUESTA MODELO:Un experimento simple: Entrena árboles con subconjuntos aleatorios del training set (bootstrap muestras diferentes)con los mismos hiperparámetros. Obtendrás árboles diferentes (distintos splits en los primeros niveles).Si calculas la métrica en el test set original para TODOS estos árboles, verás que varían.Esto demuestra que el algoritmo greedy no encontró un único óptimo global:distintos "caminos" locales conducen a distintos árboles. Un algoritmo exhaustivo (exploración completa del espacio)podría encontrar uno mejor.-->

In [ ]:
# [PhD]# TODO 6: Experimento — demostrar suboptimalidad del greedy CARTn_bootstrap = 10trees_bootstrap = []test_scores_bootstrap = []np.random.seed(RANDOM_STATE)for i in range(n_bootstrap):    # Bootstrap sample    indices = np.random.choice(len(X_train), size=len(X_train), replace=True)    X_boot = X_train.iloc[indices]    y_boot = y_train.iloc[indices]        # Entrenar árbol    tree_boot = DecisionTreeClassifier(max_depth=best_depth, random_state=i)    tree_boot.fit(X_boot, y_boot)        # Evaluar en test original    score = f1_score(y_test, tree_boot.predict(X_test), average='macro')    test_scores_bootstrap.append(score)    trees_bootstrap.append(tree_boot)test_scores_bootstrap = np.array(test_scores_bootstrap)print(f"Experimento: Entrenar {n_bootstrap} árboles en bootstrap samples distintos")print(f"\nF1-macro en test set original:")print(f"  Mean: {test_scores_bootstrap.mean():.4f}")print(f"  Std:  {test_scores_bootstrap.std():.4f}")print(f"  Min:  {test_scores_bootstrap.min():.4f}")print(f"  Max:  {test_scores_bootstrap.max():.4f}")print(f"\n✓ La variabilidad (Std ≠ 0) demuestra que CART encontró soluciones locales diferentes.")print(f"  Un algoritmo global encontraría siempre el mismo árbol óptimo.")

### E.5. Límite de Bayes: Cuando árboles profundos/superficiales son apropiadosEl **límite de Bayes** es la tasa de error teórica mínima alcanzable. - Un árbol **muy profundo** puede aproximarse al límite de Bayes si la frontera verdadera es compleja.- Un árbol **superficial** es mejor si la frontera es lineal o casi lineal.**TODO 7:** Analiza bajo qué condiciones cada uno es apropiado.<\!-- RESPUESTA MODELO:Condiciones para árboles profundos:1. La frontera de decisión es no convexa, muy compleja (multimodal).2. Las clases se solapan en el espacio original pero son separables por splits ortogonales.3. Tienes datos de entrenamiento suficientes (n >> p).Condiciones para árboles superficiales:1. La frontera de decisión es lineal o casi lineal.2. Los datos tienen mucho ruido (overfitting es un riesgo).3. Datos limitados (n cercano a p o menor).4. Necesitas interpretabilidad máxima (modelos simples).En el Wine dataset: La frontera parece moderadamente compleja (3 clases, 13 features).Un max_depth de 4-6 suele ser suficiente: más profundo → overfitting, menos profundo → underfitting.-->

In [ ]:
# [PhD]# TODO 7: Análisis de convergencia a límite de Bayes con profundidaddepths_range = np.arange(1, 15)train_scores = []test_scores = []for d in depths_range:    tree = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)    tree.fit(X_train, y_train)        train_f1 = f1_score(y_train, tree.predict(X_train), average='macro')    test_f1 = f1_score(y_test, tree.predict(X_test), average='macro')        train_scores.append(train_f1)    test_scores.append(test_f1)# Visualizarfig, ax = plt.subplots(figsize=(10, 6))ax.plot(depths_range, train_scores, 'o-', label='Train F1', linewidth=2, markersize=6)ax.plot(depths_range, test_scores, 's-', label='Test F1', linewidth=2, markersize=6)ax.axvline(best_depth, color='red', linestyle='--', alpha=0.7, label=f'Optimal depth ({best_depth})')ax.set_xlabel('max_depth')ax.set_ylabel('F1-macro')ax.set_title('Convergencia a límite de Bayes: Train vs Test por profundidad', fontsize=12)ax.legend()ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("Análisis completo.")print(f"\n✓ Se observa:")print(f"  - Train score aumenta monótonamente (árbol se ajusta mejor a train).")print(f"  - Test score aumenta, llega a máximo, y luego desciende (overfitting).")print(f"  - El óptimo está alrededor de max_depth = {best_depth}.")

### E.6. Preguntas de análisis crítico**Pregunta 1:** La importancia MDI puede estar sesgada hacia features de alta cardinalidad continua. ¿Cómo lo verificarías empíricamente en este dataset?<\!-- RESPUESTA MODELO:En Wine, todas las features son continuas. Pero podrías:1. Discretizar algunas features (binning) y ver si MDI cambia significativamente.2. Normalizar features a la misma escala y ver si el ranking cambia.3. Comparar MDI con permutation_importance (ya hecho en E.2): divergencias indican sesgos en MDI.En nuestro caso, permutation importance usualmente penaliza menos las features continuas de baja correlación.-->**Pregunta 2:** CART es greedy. Propón un experimento que demuestre que el árbol entrenado no es globalmente óptimo.<\!-- RESPUESTA MODELO:Ya lo hicimos en E.4: entrenar múltiples árboles en bootstrap samples del training set, con los mismos hiperparámetros. Si la solución fuera globalmente óptima, todos los árboles serían idénticos o muy similares. La variabilidad observada demuestra suboptimalidad local.-->**Pregunta 3:** ¿Bajo qué condiciones el límite de Bayes en este problema sería aproximado correctamente por un árbol muy profundo? ¿Y por un árbol poco profundo?<\!-- RESPUESTA MODELO:Árbol muy profundo → aproxima límite de Bayes bien si:  - Frontera verdadera es altamente no convexa.  - Dataset es grande (suficientes datos para entrenar nodos profundos).  - Número de features es moderado (no curse of dimensionality).Árbol poco profundo → aproxima bien si:  - Frontera verdadera es lineal o casi lineal.  - Dataset pequeño o ruidoso.  - Necesitas interpretabilidad.En Wine: con 13 features y 178 muestras, la frontera parece moderadamente compleja.Un árbol de profundidad 4-6 captura bien la estructura sin overfitting extremo.-->

In [ ]:
# [PhD]# 🔍 Tests de sanidad — análisis avanzadotry:    assert results.shape == (len(min_samples_split_range), len(min_samples_leaf_range)), \        "Heatmap debe tener dimensiones correctas"    assert np.all(results >= 0) and np.all(results <= 1), "Scores deben estar en [0,1]"    assert len(test_scores_bootstrap) == n_bootstrap, "Debe haber bootstrap scores para cada árbol"    assert len(train_scores) == len(test_scores) == len(depths_range), "Listas de scores deben tener igual longitud"    assert perm_df.shape[0] == 13, "Permutation importance debe tener una fila por feature"    print("✅ Tests de sanidad pasados: análisis avanzado completo")except AssertionError as e:    print(f"❌ Error: {e}")

---## Autoevaluación### 🔵 PregradoMarca las competencias que consideras haber alcanzado:- [ ] **Exploración:** Cargué el dataset, exploré shape, descripción estadística, distribución de clases.- [ ] **Preprocesamiento:** Hice train/test split estratificado correctamente (75/25).- [ ] **Modelo base:** Entrené un árbol con parámetros por defecto y evalué accuracy, F1-macro, matriz de confusión.- [ ] **Visualización:** Visualicé el árbol (primeros niveles) y los feature importances (top 10).- [ ] **Optimización:** Usé GridSearchCV para buscar max_depth óptimo (5-fold CV, métrica F1-macro).- [ ] **Comparación:** Comparé árbol vs Gaussian NB vs LDA usando validación cruzada.- [ ] **Reflexión:** Respondí las 3 preguntas de reflexión aplicada con argumentos claros.### 🟡 DoctoradoAdemás de lo anterior:- [ ] **Sensibilidad:** Construí heatmap de GridSearchCV (2D: min_samples_split vs min_samples_leaf).- [ ] **Importancia:** Calculé permutation_importance vs MDI y comparé, analizando diferencias.- [ ] **Gini analítico:** Derivé analíticamente la impureza Gini del nodo raíz y verifiqué.- [ ] **Suboptimalidad:** Diseñé experimento con bootstrap para demostrar suboptimalidad greedy.- [ ] **Límite de Bayes:** Analicé convergencia del test score con profundidad del árbol.- [ ] **Reflexión crítica:** Respondí las 3 preguntas de análisis crítico con argumentos fundamentados en teoría.### Reflexión libre¿Qué aspecto del Assignment te pareció más interesante? ¿Qué dudas te quedan?---

<\!-- NOTA PARA PRUEBA EN PAPEL:Conceptos clave a evaluar en la prueba escrita:1. Impureza Gini vs Entropía: definición, uso en CART, cálculo analítico.2. Ganancia de información (IG): cómo CART usa IG para seleccionar splits.3. Overfitting en árboles: relación con profundidad, min_samples_split, min_samples_leaf.4. Feature importance MDI vs Permutation: cuándo son similares, cuándo difieren, sesgos.5. Interpretabilidad de árboles: ventajas vs Naive Bayes, redes neuronales, SVM.6. Greedy vs Global: por qué CART no garantiza optimalidad global.Preguntas sugeridas para prueba escrita:[Pregunta 1 — Cálculo analítico]En el Wine dataset de entrenamiento, las clases tienen 59, 71 y 48 muestras respectivamente.Calcula la impureza Gini del nodo raíz. Si un split produce dos nodos con:  - Nodo izquierdo: 30 muestras (15, 10, 5 de cada clase)  - Nodo derecho: 148 muestras (44, 61, 43 de cada clase)Calcula la ganancia de información (IG) de este split. ¿Es un buen split?[Pregunta 2 — Conceptual: MDI vs PI]Explica por qué la importancia Mean Decrease in Impurity (MDI) puede estar sesgada hacia features continuas de alta cardinalidad. ¿Cómo mitigarías este sesgo?[Pregunta 3 — Aplicado]En el Wine dataset, el árbol de decisión alcanzó ~95% de accuracy en test.Un modelo completamente aleatorio alcanzaría ~33% (línea base). ¿Qué conclusión sacas sobre el problema? ¿Tiene sentido usar un árbol aquí vs otros modelos?[Pregunta 4 — Crítica: CART greedy]CART es un algoritmo greedy que selecciona splits óptimos localmente.¿Bajo qué condiciones podrías estar seguro de que el árbol entrenado es TAMBIÉN globalmente óptimo? ¿O nunca lo sabes?[Pregunta 5 — Diseño experimental]Diseña un experimento sencillo que demuestre empíricamente que CART no es globalmente óptimo (usa ideas del notebook si es necesario).Errores comunes observados en años anteriores:- Confundir feature_importance con causalidad: "Alcohol es la causa del vino tipo 2"   (NO: es correlacionado, no causante).- Olvidar estratificar el split (train/test) -> data leakage.- Usar test set para seleccionar max_depth (data leakage) en lugar de cross-validation sobre train.- Comparar Gini vs Entropía sin entender que ambos son similares; CART elige Gini por eficiencia numérica.- Pensar que "profundidad máxima del árbol = número de features" (NO: puede ser mucho más pequeño).Rúbrica de corrección:- [15 pts] Cálculo analítico de Gini e IG: precisión numérica y explicación clara.- [15 pts] Conceptos MDI/PI: entendimiento de sesgos y cuándo usarlos.- [15 pts] Interpretación de resultados: conclusiones válidas sobre el problema.- [15 pts] Crítica CART: entendimiento de limitaciones locales vs globales.- [15 pts] Diseño experimental: propuesta coherente y ejecutable.- [25 pts] Reflexión integrada: conexión entre teoría y práctica en el notebook.Total: 100 pts.-->